# FGrade Cross-Dataset Validation
## Tomato Freshness Classification — Xception, YOLOv5m, Swin Transformer

> **Purpose:** Retrains all three models on the FGrade dataset (Das et al., CVIP 2020)
> using the same hyperparameters and protocol as Dataset 1 (Enalis Kaggle).
> The 10 original ordinal classes are re-mapped to 3 freshness stages:
> - **Fresh** → classes 1–3
> - **At-risk** → classes 4–7
> - **Rotten** → classes 8–10

**Runtime:** GPU (Tesla T4) — Google Colab Pro recommended  
**Expected total time:** ~7–9 hours (all 3 models × 200 epochs)

## 0. Setup & GPU Check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [ ]:
!pip install -q timm torchmetrics seaborn scikit-learn matplotlib pandas
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 46.7 MB/s eta 0:00:00
Dependencies installed.


## 1. Download FGrade Dataset from GitHub

In [ ]:
import os

# Clone the FGrade repository
if not os.path.exists('FGrade'):
    !git clone https://github.com/skarifahmed/FGrade.git
    print('FGrade repository cloned.')
else:
    print('FGrade already present.')

# Explore structure
!find FGrade/data -type d | head -30

Cloning into 'FGrade'...
remote: Enumerating objects: 6429, done.
remote: Total 6429 (delta 0), reused 0 (delta 0), pack-reused 6429 (from 1)
Receiving objects: 100% (6429/6429), 111.25 MiB | 16.90 MiB/s, done.
Resolving deltas: 100% (1/1), done.
FGrade repository cloned.
FGrade/data
FGrade/data/Training_set
FGrade/data/Training_set/0
FGrade/data/Training_set/9
FGrade/data/Training_set/7
FGrade/data/Training_set/3
FGrade/data/Training_set/1
FGrade/data/Training_set/8
FGrade/data/Training_set/4
FGrade/data/Training_set/5
FGrade/data/Training_set/2
FGrade/data/Training_set/6
FGrade/data/Testing_set
FGrade/data/Testing_set/0
FGrade/data/Testing_set/9
FGrade/data/Testing_set/7
FGrade/data/Testing_set/3
FGrade/data/Testing_set/1
FGrade/data/Testing_set/8
FGrade/data/Testing_set/4
FGrade/data/Testing_set/5
FGrade/data/Testing_set/2
FGrade/data/Testing_set/6


## 2. Class Re-Mapping (10 → 3 classes)

In [ ]:
# Diagnostic — à lancer AVANT la cellule de re-mapping
from pathlib import Path

fgrade_path = Path('FGrade')
print("=== Structure racine FGrade ===")
for item in sorted(fgrade_path.iterdir()):
    print(f"  {item.name}/")

print("\n=== Contenu de FGrade/data (si existe) ===")
data_path = fgrade_path / 'data'
if data_path.exists():
    for item in sorted(data_path.iterdir()):
        # Compte les images dans chaque sous-dossier
        imgs = list(item.glob('**/*.jpg')) + list(item.glob('**/*.png')) + \
               list(item.glob('**/*.jpeg'))
        print(f"  {item.name}/ → {len(imgs)} images")
else:
    print("  Pas de dossier 'data'. Contenu à la racine :")
    for item in sorted(fgrade_path.iterdir()):
        if item.is_dir():
            imgs = list(item.glob('**/*.jpg')) + list(item.glob('**/*.png'))
            print(f"    {item.name}/ → {len(imgs)} images")

=== Structure racine FGrade ===
  .git/
  .gitattributes/
  LICENSE/
  README.md/
  data/
  img/
  src/

=== Contenu de FGrade/data (si existe) ===
  Testing_set/ → 1275 images
  Training_set/ → 5122 images


In [ ]:
# Diagnostic niveau 2 — structure interne des splits
from pathlib import Path

data_path = Path('FGrade/data')

for split_dir in sorted(data_path.iterdir()):
    if split_dir.is_dir():
        print(f"\n=== {split_dir.name} ===")
        for cls_dir in sorted(split_dir.iterdir()):
            if cls_dir.is_dir():
                imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')) + \
                       list(cls_dir.glob('*.jpeg')) + list(cls_dir.glob('*.JPG'))
                print(f"  {cls_dir.name}/ → {len(imgs)} images")


=== Testing_set ===
  0/ → 591 images
  1/ → 76 images
  2/ → 76 images
  3/ → 76 images
  4/ → 76 images
  5/ → 76 images
  6/ → 76 images
  7/ → 76 images
  8/ → 76 images
  9/ → 76 images

=== Training_set ===
  0/ → 2368 images
  1/ → 306 images
  2/ → 306 images
  3/ → 306 images
  4/ → 306 images
  5/ → 306 images
  6/ → 306 images
  7/ → 306 images
  8/ → 306 images
  9/ → 306 images


In [ ]:
import shutil
import random
from pathlib import Path

# ── Re-mapping corrigé : classes 0–9 (0 = plus frais, 9 = plus pourri) ────
# Classe 0 = très frais (grosse classe)
# On mappe : 0,1,2 → fresh | 3,4,5,6 → at_risk | 7,8,9 → rotten
REMAP = {
    '0': 'fresh',   '1': 'fresh',   '2': 'fresh',
    '3': 'at_risk', '4': 'at_risk', '5': 'at_risk', '6': 'at_risk',
    '7': 'rotten',  '8': 'rotten',  '9': 'rotten'
}

FGRADE_DATA_PATH = Path('FGrade/data')
TRAIN_SRC        = FGRADE_DATA_PATH / 'Training_set'
TEST_SRC         = FGRADE_DATA_PATH / 'Testing_set'
OUTPUT_PATH      = Path('fgrade_3class')

# ── Créer les dossiers de sortie ──────────────────────────────────────────
for split in ['train', 'val', 'test']:
    for cls in ['fresh', 'at_risk', 'rotten']:
        (OUTPUT_PATH / split / cls).mkdir(parents=True, exist_ok=True)

# ── Collecter les images par label fusionné ───────────────────────────────
def collect_images(src_dir, remap):
    all_images = {'fresh': [], 'at_risk': [], 'rotten': []}
    for original_class, new_label in remap.items():
        class_dir = src_dir / original_class
        if not class_dir.exists():
            print(f'WARNING: {class_dir} not found.')
            continue
        imgs = (list(class_dir.glob('*.jpg'))  +
                list(class_dir.glob('*.jpeg')) +
                list(class_dir.glob('*.png'))  +
                list(class_dir.glob('*.JPG')))
        all_images[new_label].extend(imgs)
    return all_images

train_images = collect_images(TRAIN_SRC, REMAP)
test_images  = collect_images(TEST_SRC,  REMAP)

print('Images TRAIN par classe (avant balancing):')
for cls, imgs in train_images.items():
    print(f'  {cls}: {len(imgs)}')

print('\nImages TEST par classe (avant balancing):')
for cls, imgs in test_images.items():
    print(f'  {cls}: {len(imgs)}')

Images TRAIN par classe (avant balancing):
  fresh: 2980
  at_risk: 1224
  rotten: 918

Images TEST par classe (avant balancing):
  fresh: 743
  at_risk: 304
  rotten: 228


In [ ]:
# ── Balancing + split val depuis train ────────────────────────────────────
random.seed(42)

# Balance train par undersampling au min
min_train = min(len(v) for v in train_images.values())
print(f'Balancing train à {min_train} images/classe')

# Balance test par undersampling au min
min_test = min(len(v) for v in test_images.values())
print(f'Balancing test  à {min_test} images/classe')

def copy_images(img_list, dest_dir):
    for img_path in img_list:
        dest = dest_dir / img_path.name
        # Éviter collision de noms entre classes sources différentes
        if dest.exists():
            dest = dest_dir / f"{img_path.parent.name}_{img_path.name}"
        shutil.copy2(img_path, dest)

split_counts = {}

for cls in ['fresh', 'at_risk', 'rotten']:

    # ── TEST : undersampling direct ────────────────────────────────────
    test_imgs = random.sample(test_images[cls], min_test)
    copy_images(test_imgs, OUTPUT_PATH / 'test' / cls)

    # ── TRAIN : undersample puis extraire 10% pour val ─────────────────
    sampled   = random.sample(train_images[cls], min_train)
    val_size  = int(min_train * 0.10)
    val_imgs  = sampled[:val_size]
    tr_imgs   = sampled[val_size:]

    copy_images(tr_imgs,  OUTPUT_PATH / 'train' / cls)
    copy_images(val_imgs, OUTPUT_PATH / 'val'   / cls)

    split_counts[cls] = (len(tr_imgs), len(val_imgs), len(test_imgs))

print('\nSplit final par classe (train / val / test):')
for cls, (tr, va, te) in split_counts.items():
    print(f'  {cls}: {tr} / {va} / {te}')

total = sum(sum(v) for v in split_counts.values())
print(f'\nTotal images utilisées: {total}')

Balancing train à 918 images/classe
Balancing test  à 228 images/classe

Split final par classe (train / val / test):
  fresh: 827 / 91 / 228
  at_risk: 827 / 91 / 228
  rotten: 827 / 91 / 228

Total images utilisées: 3438


## 3. Common Data Loaders & Augmentation Pipeline

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

# ── Same augmentation as Dataset 1 ───────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32   # overridden per model below
NUM_CLASSES = 3

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees=90, shear=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    # Gaussian noise approximated via Lambda
    transforms.Lambda(lambda x: x + 0.05 * torch.randn_like(x) if torch.rand(1) < 0.5 else x),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def get_loaders(batch_size):
    train_ds = datasets.ImageFolder(OUTPUT_PATH / 'train', transform=train_transforms)
    val_ds   = datasets.ImageFolder(OUTPUT_PATH / 'val',   transform=eval_transforms)
    test_ds  = datasets.ImageFolder(OUTPUT_PATH / 'test',  transform=eval_transforms)

    # Class index mapping — always print to verify
    print('Class → index mapping:', train_ds.class_to_idx)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds.class_to_idx

print('Data loaders ready.')

Data loaders ready.


## 4. Training Utilities (shared by all models)

In [ ]:
import time
import json
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 100

CLASS_NAMES = ['at_risk', 'fresh', 'rotten']  # sorted alphabetically by ImageFolder


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += images.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += images.size(0)
    return running_loss / total, correct / total


def full_train_loop(model, model_name, train_loader, val_loader,
                    criterion, optimizer, scheduler=None):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)

        if scheduler:
            scheduler.step(va_loss)

        if epoch % 20 == 0 or epoch == 1:
            elapsed = time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))
            print(f'[{model_name}] Epoch {epoch:3d}/{EPOCHS} | '
                  f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
                  f'Val Loss: {va_loss:.4f} Acc: {va_acc:.4f} | '
                  f'Elapsed: {elapsed}')

    total_time = time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))
    print(f'\n[{model_name}] Training complete. Total time: {total_time}')
    return history, total_time


@torch.no_grad()
def measure_inference_time(model, loader, n_batches=50):
    model.eval()
    times = []
    for i, (images, _) in enumerate(loader):
        if i >= n_batches:
            break
        images = images.to(DEVICE)
        start = time.perf_counter()
        _ = model(images)
        torch.cuda.synchronize()
        elapsed = (time.perf_counter() - start) / images.size(0) * 1000  # ms/img
        times.append(elapsed)
    return np.mean(times)


@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        preds  = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


def plot_and_save(history, model_name, class_names):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history['train_acc'], label='Train Accuracy')
    axes[0].plot(history['val_acc'],   label='Val Accuracy')
    axes[0].set_title(f'{model_name} — Accuracy (FGrade)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(True)

    axes[1].plot(history['train_loss'], label='Train Loss')
    axes[1].plot(history['val_loss'],   label='Val Loss')
    axes[1].set_title(f'{model_name} — Loss (FGrade)')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(f'{model_name}_fgrade_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_confusion_matrix(preds, labels, class_names, model_name):
    cm = confusion_matrix(labels, preds, normalize='true')
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='.4f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} — Confusion Matrix (FGrade)')
    plt.ylabel('True label'); plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(f'{model_name}_fgrade_cm.png', dpi=150, bbox_inches='tight')
    plt.show()


def print_metrics(preds, labels, class_names, model_name, inference_ms, train_time):
    print(f'\n{"="*60}')
    print(f'RESULTS — {model_name} on FGrade Dataset')
    print(f'{"="*60}')
    print(f'Training time     : {train_time}')
    print(f'Inference time    : {inference_ms:.1f} ms/image')
    print(f'Overall accuracy  : {accuracy_score(labels, preds)*100:.2f}%')
    print('\nPer-class report:')
    print(classification_report(labels, preds, target_names=class_names, digits=4))

    # Save as JSON for easy copy-paste into LaTeX table
    p, r, f, _ = precision_recall_fscore_support(labels, preds)
    results = {
        'model': model_name, 'dataset': 'FGrade',
        'overall_accuracy': round(accuracy_score(labels, preds)*100, 2),
        'inference_ms': round(float(inference_ms), 1),
        'training_time': train_time,
        'per_class': {
            cn: {
                'precision': round(float(p[i])*100, 2),
                'recall':    round(float(r[i])*100, 2),
                'f1':        round(float(f[i])*100, 2)
            } for i, cn in enumerate(class_names)
        }
    }
    fname = f'{model_name}_fgrade_results.json'
    with open(fname, 'w') as fp:
        json.dump(results, fp, indent=2)
    print(f'Results saved to {fname}')
    return results

print('Training utilities loaded.')

Training utilities loaded.


---
## 5. MODEL A — Xception
> Hyperparameters (same as Dataset 1): LR=1e-4, Batch=32, Dropout=0.1, Optimizer=Adam

In [ ]:
import timm

# ── Xception config ───────────────────────────────────────────────────────
XCEPTION_LR    = 1e-4
XCEPTION_BATCH = 32
XCEPTION_DROP  = 0.1

train_loader_x, val_loader_x, test_loader_x, _ = get_loaders(XCEPTION_BATCH)

xception_model = timm.create_model(
    'xception',
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_rate=XCEPTION_DROP
).to(DEVICE)

# Confirm output shape
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out   = xception_model(dummy)
    print('Xception output shape:', out.shape)  # expect [2, 3]

xception_criterion = nn.CrossEntropyLoss()
xception_optimizer = torch.optim.Adam(xception_model.parameters(), lr=XCEPTION_LR)

print('Xception model ready.')

Class → index mapping: {'at_risk': 0, 'fresh': 1, 'rotten': 2}


/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
Xception output shape: torch.Size([2, 3])
Xception model ready.


In [ ]:
# ── Train Xception ─────────────────────────────────────────────────────────
xception_history, xception_train_time = full_train_loop(
    xception_model, 'Xception',
    train_loader_x, val_loader_x,
    xception_criterion, xception_optimizer
)

# Save checkpoint
torch.save(xception_model.state_dict(), 'xception_fgrade.pth')
print('Xception checkpoint saved.')

[Xception] Epoch   1/100 | Train Loss: 0.7731 Acc: 0.6348 | Val Loss: 0.5722 Acc: 0.7619 | Elapsed: 00:00:07
[Xception] Epoch  20/100 | Train Loss: 0.1516 Acc: 0.9403 | Val Loss: 0.3535 Acc: 0.8718 | Elapsed: 00:02:09
[Xception] Epoch  40/100 | Train Loss: 0.0817 Acc: 0.9678 | Val Loss: 0.3893 Acc: 0.8901 | Elapsed: 00:04:16
[Xception] Epoch  60/100 | Train Loss: 0.0487 Acc: 0.9811 | Val Loss: 0.3312 Acc: 0.9048 | Elapsed: 00:06:24
[Xception] Epoch  80/100 | Train Loss: 0.0411 Acc: 0.9883 | Val Loss: 0.3815 Acc: 0.9048 | Elapsed: 00:08:33


In [ ]:
# ── Evaluate Xception ──────────────────────────────────────────────────────
x_preds, x_labels = get_predictions(xception_model, test_loader_x)
x_inf_time        = measure_inference_time(xception_model, test_loader_x)

plot_and_save(xception_history, 'Xception', CLASS_NAMES)
plot_confusion_matrix(x_preds, x_labels, CLASS_NAMES, 'Xception')
xception_results = print_metrics(
    x_preds, x_labels, CLASS_NAMES, 'Xception', x_inf_time, xception_train_time
)

---
## 6. MODEL B — YOLOv5m (classification variant)
> Hyperparameters: LR=1e-3, Batch=128, Dropout=0.1, Optimizer=Adam, Label smoothing=0.1

In [ ]:
# Install YOLOv5 if not already installed
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git
    !pip install -q -r yolov5/requirements.txt
    print('YOLOv5 installed.')
else:
    print('YOLOv5 already present.')

Cloning into 'yolov5'...
remote: Enumerating objects: 17960, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 17960 (delta 88), reused 49 (delta 49), pack-reused 17849 (from 3)
Receiving objects: 100% (17960/17960), 17.10 MiB | 20.20 MiB/s, done.
Resolving deltas: 100% (12221/12221), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 12.3 MB/s eta 0:00:00
YOLOv5 installed.


In [ ]:
# ── Train YOLOv5m-cls using the official classification training script ─────
# This mirrors the exact setup used for Dataset 1

YOLO_DATA_PATH = str(OUTPUT_PATH.resolve())

!python yolov5/classify/train.py \
    --model yolov5m-cls.pt \
    --data {YOLO_DATA_PATH} \
    --epochs 10 \
    --imgsz 224 \
    --batch-size 128 \
    --lr0 1e-3 \
    --label-smoothing 0.1 \
    --decay 5e-5 \
    --dropout 0.1 \
    --project runs/fgrade_yolo \
    --name yolov5m_fgrade \
    --workers 2

print('YOLOv5m training complete.')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: ERROR Error while calling W&B API: api key too short (<Response [401]>)
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
classify/train: model=yolov5m-cls.pt, data=/content/fgrade_3class, epochs=10, batch_size=128, imgsz=224, nosave=False, cache=None, device=, workers=2, project=runs/

In [ ]:
# ── Évaluation YOLOv5m sur le test set FGrade ─────────────────────────────
# On n'utilise PAS classify/val.py (il impose sa propre structure de dossiers)
# On fait l'inférence manuellement avec les DataLoaders torchvision

import torch
import sys
import time
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import json

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_PATH = Path('fgrade_3class')
CLASS_NAMES = ['at_risk', 'fresh', 'rotten']  # ordre alphabétique ImageFolder

# Add YOLOv5 directory to system path to import its modules
sys.path.insert(0, 'yolov5')

# ── Charger le meilleur checkpoint YOLOv5m ───────────────────────────────
yolo_weights = 'runs/fgrade_yolo/yolov5m_fgrade/weights/best.pt'
# Explicitly set weights_only=False to allow loading custom classes from YOLOv5 checkpoint
ckpt = torch.load(yolo_weights, map_location=DEVICE, weights_only=False)

# Le modèle est stocké dans ckpt['model'] (format Ultralytics)
yolo_model = ckpt['model'].float().eval().to(DEVICE)
print('YOLOv5m chargé depuis:', yolo_weights)
print('Paramètres:', sum(p.numel() for p in yolo_model.parameters()))

# ── DataLoader test (même transforms que l'évaluation) ───────────────────
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

test_ds     = datasets.ImageFolder(OUTPUT_PATH / 'test', transform=eval_transforms)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                         num_workers=2, pin_memory=True)

print('Class → index:', test_ds.class_to_idx)
print(f'Test samples: {len(test_ds)}')

YOLOv5m chargé depuis: runs/fgrade_yolo/yolov5m_fgrade/weights/best.pt
Paramètres: 11680483
Class → index: {'at_risk': 0, 'fresh': 1, 'rotten': 2}
Test samples: 684


In [ ]:
# ── Inférence + métriques ─────────────────────────────────────────────────
all_preds, all_labels = [], []
inference_times = []

yolo_model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        t0     = time.perf_counter()
        outputs = yolo_model(images)
        torch.cuda.synchronize()
        t1 = time.perf_counter()

        # outputs peut être un tenseur ou un objet Ultralytics
        # on extrait le tenseur de logits selon le type
        if isinstance(outputs, (list, tuple)):
            logits = outputs[0]
        else:
            logits = outputs

        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

        ms_per_img = (t1 - t0) / images.size(0) * 1000
        inference_times.append(ms_per_img)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
avg_inf_ms = np.mean(inference_times)

print(f'\nInférence terminée.')
print(f'Temps moyen par image : {avg_inf_ms:.1f} ms')
print(f'Overall accuracy      : {accuracy_score(all_labels, all_preds)*100:.2f}%')
print('\nRapport détaillé :')
print(classification_report(all_labels, all_preds,
                             target_names=CLASS_NAMES, digits=4))


Inférence terminée.
Temps moyen par image : 0.5 ms
Overall accuracy      : 80.99%

Rapport détaillé :
              precision    recall  f1-score   support

     at_risk     0.8199    0.5789    0.6787       228
       fresh     0.7604    0.9605    0.8488       228
      rotten     0.8638    0.8904    0.8769       228

    accuracy                         0.8099       684
   macro avg     0.8147    0.8099    0.8015       684
weighted avg     0.8147    0.8099    0.8015       684



In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='.4f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('YOLOv5m — Confusion Matrix (FGrade)')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig('YOLOv5m_fgrade_cm.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Sauvegarde JSON ───────────────────────────────────────────────────────
from sklearn.metrics import precision_recall_fscore_support
p, r, f, _ = precision_recall_fscore_support(all_labels, all_preds)

yolo_results = {
    'model': 'YOLOv5m', 'dataset': 'FGrade',
    'overall_accuracy': round(accuracy_score(all_labels, all_preds)*100, 2),
    'inference_ms': round(float(avg_inf_ms), 1),
    'per_class': {
        cn: {
            'precision': round(float(p[i])*100, 2),
            'recall':    round(float(r[i])*100, 2),
            'f1':        round(float(f[i])*100, 2)
        } for i, cn in enumerate(CLASS_NAMES)
    }
}

with open('YOLOv5m_fgrade_results.json', 'w') as fp:
    json.dump(yolo_results, fp, indent=2)

print('\nRésultats sauvegardés dans YOLOv5m_fgrade_results.json')
print(json.dumps(yolo_results, indent=2))


Résultats sauvegardés dans YOLOv5m_fgrade_results.json
{
  "model": "YOLOv5m",
  "dataset": "FGrade",
  "overall_accuracy": 80.99,
  "inference_ms": 0.5,
  "per_class": {
    "at_risk": {
      "precision": 81.99,
      "recall": 57.89,
      "f1": 67.87
    },
    "fresh": {
      "precision": 76.04,
      "recall": 96.05,
      "f1": 84.88
    },
    "rotten": {
      "precision": 86.38,
      "recall": 89.04,
      "f1": 87.69
    }
  }
}


---
## 7. MODEL C — Swin Transformer (Swin-T)
> Hyperparameters: LR=1e-3, Batch=128, Dropout=0.03, Optimizer=AdamW, Weight decay=1e-4, Label smoothing=0.1

In [ ]:
# ── Data loaders — identical transforms to Dataset 1 ─────────────────────
import timm
import torch
import torch.nn as nn
import math
import time
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_recall_fscore_support)

DEVICE      = torch.device('cuda')
OUTPUT_PATH = Path('fgrade_3class')
IMG_SIZE    = 224
NUM_CLASSES = 3
CLASS_NAMES = ['at_risk', 'fresh', 'rotten']   # alphabetical order (ImageFolder)

# ── Adaptable epochs — change only this value before running ──────────────
# e.g. EPOCHS = 10 for a quick test, EPOCHS = 200 for full training
EPOCHS = 10

# ── Augmentation pipeline — same as Dataset 1 ─────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees=90, shear=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(OUTPUT_PATH / 'train', transform=train_transforms)
val_ds   = datasets.ImageFolder(OUTPUT_PATH / 'val',   transform=eval_transforms)
test_ds  = datasets.ImageFolder(OUTPUT_PATH / 'test',  transform=eval_transforms)

print('class_to_idx:', train_ds.class_to_idx)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

# Batch size 64 — better gradient stability on the smaller FGrade dataset
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)


# ── Model — reinitialised from ImageNet weights ────────────────────────────
swin_v2 = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_rate=0.03,
    drop_path_rate=0.1        # stochastic depth for regularisation
).to(DEVICE)

# ── Loss with label smoothing (same as Dataset 1) ─────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ── Key fix: lower LR (1e-4 instead of 1e-3) ──────────────────────────────
# The composite 'fresh' class (original FGrade classes 0+1+2) introduces
# high intra-class variability; a high LR causes the classifier head to
# collapse onto the two more homogeneous classes (at_risk, rotten).
optimizer = torch.optim.AdamW(
    swin_v2.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ── Cosine LR scheduler with adaptive linear warmup ───────────────────────
# Warmup = 10% of total epochs, minimum 1, maximum 10
# This avoids ZeroDivisionError when EPOCHS <= WARMUP_EPOCHS
WARMUP_EPOCHS = max(1, min(10, int(EPOCHS * 0.10)))
print(f'EPOCHS: {EPOCHS} | WARMUP_EPOCHS: {WARMUP_EPOCHS}')

def lr_lambda(epoch):
    if EPOCHS <= 1:
        return 1.0                                              # no scheduling for single epoch
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS                     # linear warmup
    remaining = EPOCHS - WARMUP_EPOCHS
    if remaining == 0:
        return 1.0                                             # warmup fills all epochs
    progress = (epoch - WARMUP_EPOCHS) / remaining
    return 0.5 * (1.0 + math.cos(math.pi * progress))         # cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print('Swin v2 ready — LR=1e-4, adaptive warmup, cosine decay')
print(f'Parameters: {sum(p.numel() for p in swin_v2.parameters()):,}')

# Sanity-check output shape
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
    out   = swin_v2(dummy)
    print('Output shape:', out.shape)   # expected: [2, 3]


# ── Training loop ──────────────────────────────────────────────────────────
best_val_acc = 0.0
best_epoch   = 0
history      = {'train_loss': [], 'val_loss': [],
                'train_acc':  [], 'val_acc':  [],
                'lr':         []}

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    # Training step
    swin_v2.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = swin_v2(images)
        loss = criterion(out, labels)
        loss.backward()
        # Gradient clipping — standard practice for Vision Transformers
        torch.nn.utils.clip_grad_norm_(swin_v2.parameters(), max_norm=5.0)
        optimizer.step()
        tr_loss    += loss.item() * images.size(0)
        tr_correct += (out.argmax(1) == labels).sum().item()
        tr_total   += images.size(0)

    scheduler.step()

    # Validation step
    swin_v2.eval()
    va_loss, va_correct, va_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            out  = swin_v2(images)
            loss = criterion(out, labels)
            va_loss    += loss.item() * images.size(0)
            va_correct += (out.argmax(1) == labels).sum().item()
            va_total   += images.size(0)

    tr_acc = tr_correct / tr_total
    va_acc = va_correct / va_total
    tr_l   = tr_loss    / tr_total
    va_l   = va_loss    / va_total
    cur_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_l)
    history['val_loss'].append(va_l)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    history['lr'].append(cur_lr)

    # Save best checkpoint based on validation accuracy
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_epoch   = epoch
        torch.save(swin_v2.state_dict(), 'swin_fgrade_v2_best.pth')

    if epoch % 10 == 0 or epoch <= 15:
        elapsed = time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train {tr_acc:.4f} ({tr_l:.4f}) | '
              f'Val {va_acc:.4f} ({va_l:.4f}) | '
              f'LR {cur_lr:.2e} | '
              f'Best {best_val_acc:.4f}@ep{best_epoch} | '
              f'{elapsed}')

total_time = time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))
print(f'\nTraining complete. Best val acc: {best_val_acc:.4f} at epoch {best_epoch}')
print(f'Total training time: {total_time}')


# ── Final evaluation on test set using the BEST checkpoint ────────────────
swin_best = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=False,
    num_classes=3
).to(DEVICE)
swin_best.load_state_dict(torch.load('swin_fgrade_v2_best.pth', map_location=DEVICE))
swin_best.eval()

all_preds, all_labels = [], []
inf_times = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        t0  = time.perf_counter()
        out = swin_best(images)
        torch.cuda.synchronize()
        t1  = time.perf_counter()
        inf_times.append((t1 - t0) / images.size(0) * 1000)   # ms per image
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Verify that all three classes are represented in predictions
unique, counts = np.unique(all_preds, return_counts=True)
print('Prediction distribution:')
for u, c in zip(unique, counts):
    print(f'  {CLASS_NAMES[u]}: {c} ({c / len(all_preds) * 100:.1f}%)')

print(f'\nOverall accuracy : {accuracy_score(all_labels, all_preds) * 100:.2f}%')
print(f'Inference time   : {np.mean(inf_times):.1f} ms/image')
print('\nPer-class report:')
print(classification_report(all_labels, all_preds,
                             target_names=CLASS_NAMES, digits=4))


# ── Training curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_acc'], label='Train Accuracy')
axes[0].plot(history['val_acc'],   label='Val Accuracy')
axes[0].set_title('Swin Transformer — Accuracy (FGrade)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['train_loss'], label='Train Loss')
axes[1].plot(history['val_loss'],   label='Val Loss')
axes[1].set_title('Swin Transformer — Loss (FGrade)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True)

axes[2].plot(history['lr'])
axes[2].set_title('Learning Rate Schedule')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].grid(True)

plt.tight_layout()
plt.savefig('SwinTransformer_fgrade_v2_curves.png', dpi=150, bbox_inches='tight')
plt.show()


# ── Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='.4f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Swin Transformer — Confusion Matrix (FGrade)')
plt.ylabel('True label'); plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig('SwinTransformer_fgrade_v2_cm.png', dpi=150, bbox_inches='tight')
plt.show()


# ── Save results as JSON (ready for LaTeX table generation) ───────────────
p, r, f, _ = precision_recall_fscore_support(all_labels, all_preds)

swin_results = {
    'model': 'SwinTransformer',
    'dataset': 'FGrade',
    'overall_accuracy': round(accuracy_score(all_labels, all_preds) * 100, 2),
    'inference_ms': round(float(np.mean(inf_times)), 1),
    'training_time': total_time,
    'best_val_acc': round(best_val_acc * 100, 2),
    'best_epoch': best_epoch,
    'per_class': {
        cn: {
            'precision': round(float(p[i]) * 100, 2),
            'recall':    round(float(r[i]) * 100, 2),
            'f1':        round(float(f[i]) * 100, 2)
        }
        for i, cn in enumerate(CLASS_NAMES)
    }
}

with open('SwinTransformer_fgrade_v2_results.json', 'w') as fp:
    json.dump(swin_results, fp, indent=2)

print('\nResults saved to SwinTransformer_fgrade_v2_results.json')
print(json.dumps(swin_results, indent=2))

---
## 8. Cross-Dataset Summary Table (for LaTeX)
> Run this cell after all 3 models have been evaluated.

In [ ]:
# ── Load existing Dataset 1 results (manually input or from saved JSON) ───
dataset1_results = {
    'YOLOv5m':         {'overall_accuracy': 99.44, 'inference_ms': 156.1,
                         'per_class': {'at_risk':  {'precision': 98.33, 'recall': 98.33, 'f1': 99.15},
                                        'fresh':   {'precision': 100.0, 'recall': 100.0, 'f1': 100.0},
                                        'rotten':  {'precision': 100.0, 'recall': 100.0, 'f1': 99.17}}},
    'Xception':         {'overall_accuracy': 98.67, 'inference_ms': 762.0,
                         'per_class': {'at_risk':  {'precision': 97.70, 'recall': 99.00, 'f1': 98.35},
                                        'fresh':   {'precision': 98.60, 'recall': 98.25, 'f1': 98.42},
                                        'rotten':  {'precision': 99.68, 'recall': 98.72, 'f1': 99.20}}},
    'SwinTransformer':  {'overall_accuracy': 99.56, 'inference_ms': 33.9,
                         'per_class': {'at_risk':  {'precision': 99.12, 'recall': 98.83, 'f1': 98.97},
                                        'fresh':   {'precision': 99.72, 'recall': 99.48, 'f1': 99.60},
                                        'rotten':  {'precision': 99.67, 'recall': 99.39, 'f1': 99.53}}},
}

dataset2_results = {
    'YOLOv5m':        yolo_results,
    'Xception':       xception_results,
    'SwinTransformer': swin_results,
}

# ── Print cross-dataset summary ───────────────────────────────────────────
print('\n' + '='*70)
print('CROSS-DATASET SUMMARY — Overall Accuracy (%)')
print('='*70)
print(f'{"Model":<20} {"Dataset 1 (Enalis)":>20} {"Dataset 2 (FGrade)":>20} {"Delta":>10}')
print('-'*70)
for model in ['YOLOv5m', 'Xception', 'SwinTransformer']:
    d1_acc = dataset1_results[model]['overall_accuracy']
    d2_acc = dataset2_results[model]['overall_accuracy']
    delta  = d2_acc - d1_acc
    print(f'{model:<20} {d1_acc:>20.2f} {d2_acc:>20.2f} {delta:>+10.2f}')

print('\n\nCOPY-PASTE READY LaTeX TABLE ROW VALUES (Dataset 2):')
print('─'*70)
for model, res in dataset2_results.items():
    print(f'\n{model}:')
    print(f'  Overall accuracy : {res["overall_accuracy"]:.2f}%')
    print(f'  Inference time   : {res["inference_ms"]:.1f} ms')
    for cls in ['at_risk', 'fresh', 'rotten']:
        pc = res['per_class'][cls]
        print(f'  {cls:<10}: P={pc["precision"]:.2f}  R={pc["recall"]:.2f}  F1={pc["f1"]:.2f}')